In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Core
import os, gc, json, random, re
from pathlib import Path
from dataclasses import dataclass
from typing import List

# Data / viz
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Torch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

# Sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score


In [ ]:
# Kaggle dataset paths
DATA_DIR   = Path("/kaggle/input/nlp-getting-started")
TRAIN_CSV  = DATA_DIR/"train.csv"
TEST_CSV   = DATA_DIR/"test.csv"
SAMPLE_SUB = DATA_DIR/"sample_submission.csv"

# Columns
ID_COL     = "id"
TEXT_COL   = "text"
KW_COL     = "keyword"
LOC_COL    = "location"
TARGET_COL = "target"

# CV & seeds
NUM_FOLDS   = 5
RANDOM_SEED = 42

# Vocab & sequence
MAX_VOCAB = 40000
MIN_FREQ  = 2
SEQ_LEN   = 40   # good for tweets

# Training
BATCH_SIZE   = 512      # lower to 256 if OOM
EPOCHS       = 8
EARLY_STOP   = 2
BASE_LR      = 2e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 2

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("CUDA:", torch.version.cuda, "| cuDNN:", torch.backends.cudnn.version())
    print("GPU:", torch.cuda.get_device_name(0))

# Output dirs
OUT_DIR  = Path("/kaggle/working"); OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = OUT_DIR/"plots"; PLOT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def set_seed(s=RANDOM_SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

def clean_text(t: str) -> str:
    t = str(t).lower()
    t = re.sub(r"http\S+", " ", t)                 # URLs
    t = re.sub(r"[^a-z0-9'#\s]+", " ", t)          # keep letters/digits/'/#
    t = re.sub(r"\s+", " ", t).strip()
    return t

def simple_tokenize(t: str) -> List[str]:
    return clean_text(t).split()


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Use keyword by prepending it to text
train_df[KW_COL] = train_df[KW_COL].fillna("").str.replace("%20", " ", regex=False)
test_df[KW_COL]  = test_df[KW_COL].fillna("").str.replace("%20", " ", regex=False)

train_df["text_full"] = (train_df[KW_COL] + " " + train_df[TEXT_COL]).str.strip()
test_df ["text_full"] = (test_df [KW_COL] + " " + test_df [TEXT_COL]).str.strip()

train_df["text_clean"] = train_df["text_full"].map(clean_text)
test_df ["text_clean"] = test_df ["text_full"].map(clean_text)

display(train_df.head(3)[[TEXT_COL, KW_COL, "text_clean", TARGET_COL]])
print("Train rows:", len(train_df), "Test rows:", len(test_df))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

train_df["len_tokens"] = train_df["text_clean"].str.split().map(len)
axes[0].hist(train_df["len_tokens"], bins=40)
axes[0].axvline(SEQ_LEN, linestyle="--")
axes[0].set_title("Token length distribution")

train_df[TARGET_COL].value_counts().sort_index().plot(kind="bar", ax=axes[1])
axes[1].set_title("Class balance 0 non disaster 1 disaster")
plt.tight_layout(); plt.show()


Token length (left): Tweets cluster around ~12–20 tokens with a short tail; almost all are < 40 tokens. The dashed line at SEQ_LEN = 40 means our padding/truncation setting will cover >99% of tweets with minimal information loss while keeping batches fast and memory-friendly.

Class balance (right): There are more non-disaster (0) than disaster (1) tweets (roughly 57/43). That’s only a mild imbalance: a naïve “always 0” baseline would score ~0.57 accuracy but 0.0 F1 for class 1, so optimizing F1 and using stratified CV + class-weighted loss / threshold tuning is appropriate.

In [ ]:
num_classes = 2
train_df["label_id"] = train_df[TARGET_COL].astype(int)


In [ ]:
from collections import Counter

def build_vocab(texts: List[str], max_vocab=MAX_VOCAB, min_freq=MIN_FREQ):
    counter = Counter()
    for t in texts: counter.update(simple_tokenize(t))
    stoi = {"<pad>":0, "<unk>":1}
    for tok, freq in counter.most_common():
        if freq < min_freq: continue
        if tok in stoi:     continue
        stoi[tok] = len(stoi)
        if len(stoi) >= max_vocab: break
    itos = {i:s for s,i in stoi.items()}
    return stoi, itos

stoi, itos = build_vocab(train_df["text_clean"].tolist())
vocab_size = len(stoi)
print("Vocab size:", vocab_size)

def encode_text(t: str, seq_len: int=SEQ_LEN):
    toks = simple_tokenize(t)
    ids = [stoi.get(tok, 1) for tok in toks]   # 1=<unk>
    if len(ids) >= seq_len: ids = ids[:seq_len]
    else: ids += [0]*(seq_len - len(ids))      # 0=<pad>
    return np.array(ids, dtype=np.int64)


In [ ]:
class TextDataset(Dataset):
    def __init__(self, df: pd.DataFrame, has_label=True):
        self.ids = df[ID_COL].values
        self.texts = df["text_clean"].values
        self.labels = df["label_id"].values if has_label else None
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        x = encode_text(self.texts[idx], SEQ_LEN)
        l = max(1, int((x != 0).sum()))  # clamp to avoid 0-length
        if self.labels is None:
            return torch.tensor(x), torch.tensor(l), self.ids[idx]
        y = int(self.labels[idx])
        return torch.tensor(x), torch.tensor(l), torch.tensor(y), self.ids[idx]

def collate_train(batch):
    xs, ls, ys, ids = zip(*batch)
    return torch.stack(xs), torch.stack(ls), torch.tensor(ys), torch.tensor(ids)

def collate_test(batch):
    xs, ls, ids = zip(*batch)
    return torch.stack(xs), torch.stack(ls), torch.tensor(ids)


In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden=128, layers=1, dropout=0.2, kind="lstm", bidirectional=False, num_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.kind = kind.lower(); self.bidir = bidirectional
        rnn_cls = nn.LSTM if self.kind=="lstm" else nn.GRU
        self.rnn = rnn_cls(embed_dim, hidden, num_layers=layers, batch_first=True,
                           dropout=dropout if layers>1 else 0, bidirectional=bidirectional)
        mult = 2 if bidirectional else 1
        self.head = nn.Sequential(
            nn.LayerNorm(hidden*mult),
            nn.Linear(hidden*mult, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x, lengths):
        emb = self.embed(x)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, h = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        h_n = h[0] if self.kind=="lstm" else h
        last = torch.cat([h_n[-2], h_n[-1]], dim=1) if self.bidir else h_n[-1]
        return self.head(last)

class BiLSTMWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden=96, layers=1, dropout=0.3, num_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.LSTM(embed_dim, hidden, num_layers=layers, batch_first=True,
                           bidirectional=True, dropout=dropout if layers>1 else 0)
        self.att_w = nn.Linear(hidden*2, hidden*2)
        self.att_v = nn.Linear(hidden*2, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden*2, num_classes)
    def forward(self, x, lengths):
        emb = self.embed(x)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        e = torch.tanh(self.att_w(out)); scores = self.att_v(e).squeeze(-1)
        scores_fp32 = scores.float(); scores_fp32.masked_fill_(x == 0, -1e4)
        att = torch.softmax(scores_fp32, dim=1).to(out.dtype)
        ctx = torch.bmm(att.unsqueeze(1), out).squeeze(1)
        return self.head(self.dropout(ctx))

# Improved v2 (spatial dropout + layernorm)
class SpatialDropout1D(nn.Dropout2d):
    def forward(self, x):  # x: (B,T,E)
        x = x.unsqueeze(2)   # (B,T,1,E)
        x = super().forward(x)
        return x.squeeze(2)

class BiLSTMWithAttentionV2(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden=112, layers=1, dropout=0.28, emb_spatial_drop=0.15, num_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.emb_drop = SpatialDropout1D(p=emb_spatial_drop)
        self.rnn = nn.LSTM(embed_dim, hidden, num_layers=layers, batch_first=True,
                           bidirectional=True, dropout=dropout if layers>1 else 0)
        self.att_w = nn.Linear(hidden*2, hidden*2)
        self.att_v = nn.Linear(hidden*2, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden*2)
        self.head = nn.Linear(hidden*2, num_classes)
    def forward(self, x, lengths):
        emb = self.embed(x); emb = self.emb_drop(emb)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        e = torch.tanh(self.att_w(out)); scores = self.att_v(e).squeeze(-1)
        scores_fp32 = scores.float(); scores_fp32.masked_fill_(x == 0, -1e4)
        att = torch.softmax(scores_fp32, dim=1).to(out.dtype)
        ctx = torch.bmm(att.unsqueeze(1), out).squeeze(1)
        ctx = self.norm(ctx)
        return self.head(self.dropout(ctx))


In [ ]:
@dataclass
class TrainCfg:
    lr: float = BASE_LR
    epochs: int = EPOCHS
    early: int = EARLY_STOP
    weight_decay: float = WEIGHT_DECAY

scaler = GradScaler('cuda', enabled=(DEVICE=='cuda'))

def train_epoch(model, loader, opt, loss_fn):
    model.train(); losses=[]
    for x, l, y, _ids in loader:
        x = x.to(DEVICE, non_blocking=True)
        l = l.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        if DEVICE == 'cuda':
            with autocast('cuda', enabled=True):
                logits = model(x, l)
                loss = loss_fn(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        else:
            logits = model(x, l); loss = loss_fn(logits, y)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        losses.append(loss.item())
    return float(np.mean(losses))

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    all_y, all_p = [], []
    for x, l, y, _ids in loader:
        x = x.to(DEVICE, non_blocking=True)
        l = l.to(DEVICE, non_blocking=True)
        if DEVICE == 'cuda':
            with autocast('cuda', enabled=True):
                logits = model(x, l)
        else:
            logits = model(x, l)
        preds = logits.argmax(1).cpu().numpy()
        all_p.append(preds); all_y.append(y.numpy())
    y = np.concatenate(all_y) if all_y else np.array([])
    p = np.concatenate(all_p) if all_p else np.array([])
    if len(y)==0: return np.nan, np.nan
    return accuracy_score(y, p), f1_score(y, p, average="macro")

# Variant with OneCycleLR
def train_epoch_sched(model, loader, opt, loss_fn, scheduler):
    model.train(); losses=[]
    for x, l, y, _ids in loader:
        x = x.to(DEVICE, non_blocking=True)
        l = l.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        if DEVICE == 'cuda':
            with autocast('cuda', enabled=True):
                logits = model(x, l)
                loss = loss_fn(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        else:
            logits = model(x, l); loss = loss_fn(logits, y)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        scheduler.step()
        losses.append(loss.item())
    return float(np.mean(losses))


In [ ]:
def safe_compile(model):
    if DEVICE != "cuda": return model
    major, minor = torch.cuda.get_device_capability()
    if major < 7:
        print(f"Skipping torch.compile (GPU cc {major}.{minor} < 7.0).")
        return model
    try:
        return torch.compile(model)
    except Exception as e:
        print("torch.compile skipped due to:", repr(e))
        return model


In [ ]:
# Base set (3 models to satisfy assignment requirement)
model_space = [
    dict(name="LSTM_small",   typ="plain", args=dict(embed_dim=128, hidden=128, layers=1, dropout=0.2, kind="lstm", bidirectional=False)),
    dict(name="GRU_medium",   typ="plain", args=dict(embed_dim=128, hidden=192, layers=2, dropout=0.3, kind="gru",  bidirectional=False)),
    dict(name="BiLSTM_attn",  typ="attn",  args=dict(embed_dim=128, hidden=96,  layers=1, dropout=0.3)),
]

# Improved set (optional, tends to score a bit better)
model_space_improved = [
    dict(name="BiLSTM_attn_v2_112d28", typ="attn_v2", args=dict(embed_dim=128, hidden=112, layers=1, dropout=0.28, emb_spatial_drop=0.15)),
    dict(name="BiLSTM_attn_v2_128d30", typ="attn_v2", args=dict(embed_dim=128, hidden=128, layers=1, dropout=0.30, emb_spatial_drop=0.12)),
    dict(name="BiLSTM_attn_v2_128d35", typ="attn_v2", args=dict(embed_dim=128, hidden=128, layers=1, dropout=0.35, emb_spatial_drop=0.12)),
]

display(pd.DataFrame(model_space))


In [ ]:
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=RANDOM_SEED)

cv_rows = []
fold_hist = {}

for conf in model_space:
    print(f"\nModel {conf['name']}")
    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label_id"]), start=1):
        tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
        va_df = train_df.iloc[va_idx].reset_index(drop=True)

        tr_ds = TextDataset(tr_df, has_label=True)
        va_ds = TextDataset(va_df, has_label=True)
        tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                               collate_fn=collate_train, num_workers=NUM_WORKERS, pin_memory=True)
        va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,
                               collate_fn=collate_train, num_workers=NUM_WORKERS, pin_memory=True)

        # Build model
        if conf["typ"]=="plain":
            model = RNNClassifier(vocab_size=vocab_size, num_classes=num_classes, **conf["args"]).to(DEVICE)
        else:
            model = BiLSTMWithAttention(vocab_size=vocab_size, num_classes=num_classes, **conf["args"]).to(DEVICE)
        model = safe_compile(model)

        # Weighted loss (mild imbalance)
        class_counts = tr_df["label_id"].value_counts().reindex([0,1]).fillna(0).values
        cw = torch.tensor(1.0 / np.clip(class_counts, 1, None), dtype=torch.float32)
        cw = (cw / cw.mean()).to(DEVICE)
        loss_fn = nn.CrossEntropyLoss(weight=cw)

        opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

        best_f1, best_state, bad = -1.0, None, 0
        hist = []
        for epoch in range(1, EPOCHS+1):
            tr_loss = train_epoch(model, tr_loader, opt, loss_fn)
            acc, f1m = eval_epoch(model, va_loader)
            hist.append((epoch, tr_loss, acc, f1m))
            print(f" fold {fold_idx:02d} epoch {epoch:02d} loss {tr_loss:.4f} acc {acc:.4f} f1 {f1m:.4f}")
            if f1m > best_f1 + 1e-6 or best_state is None:
                best_f1, best_state, bad = f1m, {k:v.detach().cpu() for k,v in model.state_dict().items()}, 0
            else:
                bad += 1
            if bad >= EARLY_STOP: break

        if best_state is not None: model.load_state_dict(best_state)
        acc, f1m = eval_epoch(model, va_loader)
        cv_rows.append({"model": conf["name"], "fold": fold_idx, "val_acc": float(acc), "val_f1_macro": float(f1m)})
        fold_hist[(conf["name"], fold_idx)] = pd.DataFrame(hist, columns=["epoch","train_loss","val_acc","val_f1_macro"])

        del model, tr_loader, va_loader, tr_ds, va_ds
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

cv_df = pd.DataFrame(cv_rows)
summary = cv_df.groupby("model").agg(
    mean_f1=("val_f1_macro","mean"),
    std_f1=("val_f1_macro","std"),
    mean_acc=("val_acc","mean"),
    std_acc=("val_acc","std")
).sort_values("mean_f1", ascending=False)

display(summary)
best_name = summary.index[0]
print("Best by CV (F1 macro):", best_name)


In [ ]:
best_model_name = best_name if 'best_name' in locals() else summary.index[0]
pick_key = next(k for k in fold_hist if k[0]==best_model_name)
hist = fold_hist[pick_key]
plt.figure(figsize=(6,4))
plt.plot(hist["epoch"], hist["train_loss"], label="train_loss")
plt.plot(hist["epoch"], hist["val_f1_macro"], label="val_f1_macro")
plt.xlabel("epoch"); plt.ylabel("loss / F1")
plt.title(f"{best_model_name} • fold {pick_key[1]}")
plt.legend(); plt.tight_layout()
plt.savefig(PLOT_DIR/f"learning_{best_model_name}_fold{pick_key[1]}.png", dpi=150)
plt.show()


Insights from the learning curve (BiLSTM_attn, fold 1)

Healthy optimization: training loss drops smoothly every epoch → the model is learning; no instability or vanishing gradients.

Fast early gains: validation F1 jumps from ~0.68 → ~0.74–0.75 by epoch 2–3, meaning most of the signal is captured quickly.

Plateau & mild overfit: after ~epoch 5–7 the orange curve flattens and nudges down at epoch 8 while train loss keeps falling → diminishing returns and the start of overfitting.

Good capacity (not underfitting): since F1 plateaus with loss still decreasing, capacity is sufficient; more epochs won’t help generalization.

Best checkpoint window: save/stop around epochs 5–7 (your patience=2 will naturally pick this region).

In [ ]:
# Skip if you don't want extra tuning
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=RANDOM_SEED)

cv_rows2, fold_hist2 = [], {}

for conf in model_space_improved:
    print(f"\nModel {conf['name']}")
    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label_id"]), start=1):
        tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
        va_df = train_df.iloc[va_idx].reset_index(drop=True)

        tr_ds = TextDataset(tr_df, has_label=True)
        va_ds = TextDataset(va_df, has_label=True)
        tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                               collate_fn=collate_train, num_workers=NUM_WORKERS, pin_memory=True)
        va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,
                               collate_fn=collate_train, num_workers=NUM_WORKERS, pin_memory=True)

        model = BiLSTMWithAttentionV2(vocab_size=vocab_size, num_classes=num_classes, **conf["args"]).to(DEVICE)
        model = safe_compile(model)

        loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=BASE_LR, steps_per_epoch=len(tr_loader), epochs=EPOCHS, pct_start=0.25)

        best_f1, best_state, bad = -1.0, None, 0
        hist = []
        for epoch in range(1, EPOCHS+1):
            tr_loss = train_epoch_sched(model, tr_loader, opt, loss_fn, sched)
            acc, f1m = eval_epoch(model, va_loader)
            hist.append((epoch, tr_loss, acc, f1m))
            print(f" fold {fold_idx:02d} epoch {epoch:02d} loss {tr_loss:.4f} acc {acc:.4f} f1 {f1m:.4f}")
            if f1m > best_f1 + 1e-6 or best_state is None:
                best_f1, best_state, bad = f1m, {k:v.detach().cpu() for k,v in model.state_dict().items()}, 0
            else:
                bad += 1
            if bad >= EARLY_STOP: break

        if best_state is not None: model.load_state_dict(best_state)
        acc, f1m = eval_epoch(model, va_loader)
        cv_rows2.append({"model": conf["name"], "fold": fold_idx, "val_acc": float(acc), "val_f1_macro": float(f1m)})
        fold_hist2[(conf["name"], fold_idx)] = pd.DataFrame(hist, columns=["epoch","train_loss","val_acc","val_f1_macro"])

        del model, tr_loader, va_loader, tr_ds, va_ds
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

if len(cv_rows2):
    cv_df2 = pd.DataFrame(cv_rows2)
    summary2 = cv_df2.groupby("model").agg(
        mean_f1=("val_f1_macro","mean"),
        std_f1=("val_f1_macro","std"),
        mean_acc=("val_acc","mean"),
        std_acc=("val_acc","std")
    ).sort_values("mean_f1", ascending=False)
    display(summary2)
    best_name2 = summary2.index[0]
    print("Best (improved) by F1:", best_name2)


In [ ]:
# Choose best between base (summary) and improved (summary2 if exists)
final_name = best_name
final_space = model_space
final_typ = None
final_args = None

if 'summary2' in locals() and len(summary2) > 0:
    base_top = summary.loc[best_name, "mean_f1"]
    imp_top  = summary2.loc[summary2.index[0], "mean_f1"]
    if imp_top >= base_top:
        final_name = summary2.index[0]
        final_space = model_space_improved

final_conf = next(c for c in (model_space + model_space_improved) if c["name"] == final_name)
print("Final picked config:", final_conf)


In [ ]:
full_df = train_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
cut = int(len(full_df) * 0.98)
tr_full = full_df.iloc[:cut].reset_index(drop=True)
va_tail = full_df.iloc[cut:].reset_index(drop=True)

tr_ds = TextDataset(tr_full, has_label=True)
va_ds = TextDataset(va_tail, has_label=True)
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       collate_fn=collate_train, num_workers=NUM_WORKERS, pin_memory=True)
va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,
                       collate_fn=collate_train, num_workers=NUM_WORKERS, pin_memory=True)

# Build model
if final_conf["typ"] == "plain":
    model = RNNClassifier(vocab_size=vocab_size, num_classes=2, **final_conf["args"]).to(DEVICE)
elif final_conf["typ"] == "attn":
    model = BiLSTMWithAttention(vocab_size=vocab_size, num_classes=2, **final_conf["args"]).to(DEVICE)
else:
    model = BiLSTMWithAttentionV2(vocab_size=vocab_size, num_classes=2, **final_conf["args"]).to(DEVICE)

model = safe_compile(model)

# Loss (class weights)
class_counts = tr_full["label_id"].value_counts().reindex([0,1]).fillna(0).values
cw = torch.tensor(1.0 / np.clip(class_counts, 1, None), dtype=torch.float32)
cw = (cw / cw.mean()).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=cw)

opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

best_f1, best_state, bad = -1.0, None, 0
for epoch in range(1, EPOCHS+1):
    tr_loss = train_epoch(model, tr_loader, opt, loss_fn)
    acc, f1m = eval_epoch(model, va_loader)
    print(f" full epoch {epoch:02d} | loss {tr_loss:.4f} | tail_acc {acc:.4f} | tail_f1 {f1m:.4f}")
    if f1m > best_f1 + 1e-6 or best_state is None:
        best_f1, best_state, bad = f1m, {k:v.detach().cpu() for k,v in model.state_dict().items()}, 0
    else:
        bad += 1
    if bad >= EARLY_STOP: break

if best_state is not None: model.load_state_dict(best_state)
acc_tail, f1_tail = eval_epoch(model, va_loader)
print(f"Tail holdout — Acc {acc_tail:.4f} | F1 {f1_tail:.4f}")

# Tune thresholds on tail
@torch.no_grad()
def collect_probs(model, loader):
    model.eval(); probs, ys = [], []
    for x, l, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True); l = l.to(DEVICE, non_blocking=True)
        with autocast('cuda', enabled=(DEVICE=='cuda')):
            logits = model(x, l); p = torch.softmax(logits, dim=1)[:, 1]
        probs.append(p.cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys)

va_probs, va_true = collect_probs(model, va_loader)
ths = np.linspace(0.20, 0.80, 601)
best_thr_f1, best_f1_score = 0.50, -1
best_thr_acc, best_acc     = 0.50, -1
for t in ths:
    pred = (va_probs >= t).astype(int)
    f1m  = f1_score(va_true, pred, average="macro")
    acc  = accuracy_score(va_true, pred)
    if f1m > best_f1_score: best_f1_score, best_thr_f1 = f1m, t
    if acc  > best_acc:      best_acc,      best_thr_acc = acc, t

print(f"\nTail @0.50 -> Acc {accuracy_score(va_true,(va_probs>=0.5)):.4f} | F1 {f1_score(va_true,(va_probs>=0.5),average='macro'):.4f}")
print(f"Best F1  -> t={best_thr_f1:.3f} | F1={best_f1_score:.4f}")
print(f"Best Acc -> t={best_thr_acc:.3f} | Acc={best_acc:.4f}")

THR_F1  = float(best_thr_f1)
THR_ACC = float(best_thr_acc)

# Save checkpoint
torch.save({
    "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "vocab_size": vocab_size, "stoi": stoi, "itos": itos,
    "config": final_conf, "tail_acc": float(acc_tail), "tail_f1": float(f1_tail),
    "thr_f1": THR_F1, "thr_acc": THR_ACC
}, OUT_DIR/"best_text_rnn.pt")


On my small tail hold-out set the model scored 82.35% accuracy and 0.8216 F1 using the default 0.50 cutoff. Since 0.50 is arbitrary, I swept thresholds and found t = 0.732 gave the best F1 (0.8364) and also the best accuracy (0.8431). The higher cutoff makes the classifier more conservative, which reduced false positives more than true positives, so both metrics improved. The tail set is tiny so the exact threshold may vary a bit, but I used t=0.732 for test predictions because Kaggle scores F1.

In [ ]:
USE_ACC_THRESHOLD = False
thr = THR_ACC if USE_ACC_THRESHOLD else THR_F1
print("Using decision threshold:", thr)

test_ds = TextDataset(test_df, has_label=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_test, num_workers=NUM_WORKERS, pin_memory=True)

@torch.no_grad()
def predict_test_probs(model, loader):
    model.eval(); all_ids, all_probs = [], []
    for x, l, ids in loader:
        x = x.to(DEVICE, non_blocking=True); l = l.to(DEVICE, non_blocking=True)
        with autocast('cuda', enabled=(DEVICE=='cuda')):
            logits = model(x, l); p = torch.softmax(logits, dim=1)[:, 1]
        all_ids.append(ids.numpy()); all_probs.append(p.cpu().numpy())
    return np.concatenate(all_ids), np.concatenate(all_probs)

ids, probs = predict_test_probs(model, test_loader)
preds = (probs >= thr).astype(int)

sample = pd.read_csv(SAMPLE_SUB)
submission = sample[[ID_COL]].merge(pd.DataFrame({ID_COL: ids, TARGET_COL: preds}), on=ID_COL, how="left")
submission[TARGET_COL] = submission[TARGET_COL].astype(int)

sub_path = OUT_DIR/"submission.csv"
submission.to_csv(sub_path, index=False)
print("Wrote:", sub_path, "| Positive rate:", submission[TARGET_COL].mean())
display(submission.head())
